<a href="https://colab.research.google.com/github/hannahbakergomez-dotcom/Fintrackfinalproject/blob/main/HBG_modelling_sessions_13_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SESSION 13 & 14 — LOGISTIC REGRESSION + TIME SERIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller


# ============================================================
# PART 1 — SIGMOID FUNCTION
# ============================================================

x = np.linspace(-20, 20, 300)
sigmoid = 1 / (1 + np.exp(-x))

plt.figure(figsize=(10, 5))
plt.plot(x, sigmoid)
plt.scatter(0, 0.5, label="x = 0, probability = 0.5")
plt.title("Sigmoid Function")
plt.xlabel("x")
plt.ylabel("Probability")
plt.legend()
plt.show()


# ============================================================
# PART 2 — LOGISTIC REGRESSION WITH SYNTHETIC DATA
# ============================================================

np.random.seed(42)

exam_1 = np.random.normal(50, 10, 120)
exam_2 = np.random.normal(50, 10, 120)

noise = np.random.normal(0, 5, 120)
admitted = (exam_1 + exam_2 + noise > 100).astype(int)

X_scores = np.column_stack((exam_1, exam_2))
y_scores = admitted

admission_model = LogisticRegression()
admission_model.fit(X_scores, y_scores)

print("Admission model coefficients:")
print("Intercept:", admission_model.intercept_[0])
print("Coefficients:", admission_model.coef_[0])

sample_students = np.array([
    [100, 100],
    [10, 10],
    [50, 50],
    [48, 48]
])

probabilities = admission_model.predict_proba(sample_students)[:, 1]

print("\nAdmission probability examples:")
for scores, probability in zip(sample_students, probabilities):
    print(scores, "->", round(probability * 100, 2), "%")

x_line = np.array([exam_1.min(), exam_1.max()])
y_line = -(admission_model.intercept_[0] + admission_model.coef_[0][0] * x_line) / admission_model.coef_[0][1]

plt.figure(figsize=(10, 6))
plt.scatter(exam_1[y_scores == 1], exam_2[y_scores == 1], label="Admitted")
plt.scatter(exam_1[y_scores == 0], exam_2[y_scores == 0], label="Not admitted")
plt.plot(x_line, y_line, label="Decision boundary")
plt.xlabel("Exam 1 Score")
plt.ylabel("Exam 2 Score")
plt.title("Logistic Regression Decision Boundary")
plt.legend()
plt.show()


# ============================================================
# PART 3 — TITANIC CLASSIFICATION EXAMPLE
# ============================================================

# Download Titanic data
url = "https://raw.githubusercontent.com/itb-ie/titanic/main/train.csv"
response = requests.get(url)

with open("titanic.csv", "w") as file:
    file.write(response.text)

titanic = pd.read_csv("titanic.csv")

# Fill missing values
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].mean())
titanic["Fare"] = titanic["Fare"].fillna(titanic["Fare"].mean())
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])

# Create dummy variables
titanic = pd.get_dummies(
    titanic,
    columns=["Sex", "Embarked"],
    drop_first=True
)

# Drop unnecessary columns
titanic = titanic.drop(
    ["PassengerId", "Name", "Ticket", "Cabin"],
    axis=1
)

# Scale numerical variables
scaler = StandardScaler()
titanic[["Age", "Fare"]] = scaler.fit_transform(titanic[["Age", "Fare"]])

# Separate features and target
X = titanic.drop("Survived", axis=1)
y = titanic["Survived"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train model
titanic_model = LogisticRegression(max_iter=1000)
titanic_model.fit(X_train, y_train)

# Predict
y_pred = titanic_model.predict(X_test)

# Evaluate
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [accuracy, precision, recall, f1]
})

print("\nTitanic Confusion Matrix:")
print(cm)

print("\nTitanic Model Metrics:")
print(metrics)


# ============================================================
# PART 4 — TIME SERIES WITH STOCK DATA
# ============================================================

import yfinance as yf

ticker = "AAPL"
stock_data = yf.download(
    ticker,
    start="2020-01-01",
    end="2025-12-31",
    auto_adjust=True
)

stock_prices = stock_data["Close"]

plt.figure(figsize=(12, 5))
plt.plot(stock_prices)
plt.title("Apple Stock Price Time Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

# Seasonal decomposition
decomposition = seasonal_decompose(
    stock_prices.dropna(),
    model="additive",
    period=365
)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(decomposition.observed)
axes[0].set_title("Original Series")

axes[1].plot(decomposition.trend)
axes[1].set_title("Trend")

axes[2].plot(decomposition.seasonal)
axes[2].set_title("Seasonality")

axes[3].plot(decomposition.resid)
axes[3].set_title("Noise / Residuals")

plt.tight_layout()
plt.show()


# ============================================================
# PART 5 — TREND REGRESSION
# ============================================================

trend = decomposition.trend.dropna()

X_trend = np.arange(len(trend)).reshape(-1, 1)
y_trend = trend.values

trend_model = LinearRegression()
trend_model.fit(X_trend, y_trend)

trend_prediction = trend_model.predict(X_trend)

slope = trend_model.coef_[0]
intercept = trend_model.intercept_

print("\nTrend Regression:")
print("Slope:", slope)
print("Intercept:", intercept)

plt.figure(figsize=(12, 5))
plt.plot(trend.index, trend, label="Trend")
plt.plot(trend.index, trend_prediction, label="Linear trend")
plt.title("Linear Regression on Trend Component")
plt.legend()
plt.show()


# ============================================================
# PART 6 — STATIONARITY TEST
# ============================================================

adf_result = adfuller(stock_prices.dropna())

print("\nADF Test:")
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])

if adf_result[1] < 0.05:
    print("The series is likely stationary.")
else:
    print("The series is likely non-stationary.")

## Explanation of the Code/notes

This notebook focuses on logistic regression and time series analysis.

The first part introduces the sigmoid function. The sigmoid function transforms any numerical value into a probability between 0 and 1. This is important because logistic regression predicts probabilities before converting them into classes.

The second part applies logistic regression to synthetic student admission data. Two exam scores are used as explanatory variables, and the model predicts whether a student is admitted or not admitted. The decision boundary shows the line that separates the two predicted classes.

The third part applies logistic regression to the Titanic dataset. The goal is to predict whether a passenger survived based on variables such as age, fare, class, sex, and port of embarkation.

Before training the model, the data is preprocessed. Missing values are filled, categorical variables are converted into dummy variables, unnecessary columns are removed, and numerical variables are standardized.

The Titanic model is evaluated using a confusion matrix, accuracy, precision, recall, and F1 score. These metrics help measure how well the classification model predicts survival.

The fourth part introduces time series analysis using Apple stock price data. A time series is data ordered over time, so the date order matters.

The code decomposes the time series into trend, seasonality, and residual noise. The trend shows the long-term movement, seasonality shows repeated patterns, and residuals show unexplained variation.

The fifth part applies linear regression to the trend component. This helps estimate whether the general direction of the series is increasing or decreasing over time.

Finally, the Augmented Dickey-Fuller test is used to check stationarity. A stationary time series has statistical properties that remain stable over time. If the p-value is below 0.05, the series is likely stationary; otherwise, it is likely non-stationary.